In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
lakshmi25npathi_imdb_dataset_of_50k_movie_reviews_path = kagglehub.dataset_download('lakshmi25npathi/imdb-dataset-of-50k-movie-reviews')

print('Data source import complete.')


# **Part 1: Dataset Preparation and Fine-Tuning**

In [ ]:
!pip install transformers datasets evaluate

In [ ]:
import pandas as pd

# Load the dataset from Kaggle
file_path = "/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"
df = pd.read_csv(file_path)

# Display first few rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


*Step 2: Data Preprocessing*

In [ ]:
print(df['sentiment'].unique())  # Check all unique values in the sentiment column

[1 0]


**Convert Data to Dataset Format**

**Load Pre-trained Model**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer

# Load dataset
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

# Encode labels (positive -> 1, negative -> 0)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

df.info()
df['sentiment'].value_counts()  #class distribution

# Keep only necessary columns
df = df[['review', 'sentiment']]
df.head()



# Split into train, validation, test
train_texts, temp_texts, train_labels, temp_labels = train_test_split(df['review'], df['sentiment'], test_size=0.2, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)
# Verify splits
print(f"Training set size: {len(train_texts)}")
print(f"Validation set size: {len(eval_texts)}")
print(f"Test set size: {len(test_texts)}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 781.4+ KB
Training set size: 40000
Validation set size: 5000
Test set size: 5000


*Step 3: Model Selection and Tokenization*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer


# Tokenization
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=256)

print(train_encodings.keys())

dict_keys(['input_ids', 'attention_mask'])


In [ ]:
from transformers import DistilBertForSequenceClassification

# Load model
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=1)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Convert labels to torch tensors (ensure correct type)
import torch

class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)  # Ensure labels are single integers
        return item

# Create datasets
train_dataset = IMDBDataset(train_encodings, list(train_labels))
val_dataset = IMDBDataset(val_encodings, list(val_labels))
test_dataset = IMDBDataset(test_encodings, list(test_labels))
print(train_dataset[0])
print(test_dataset[0])

{'input_ids': tensor([  101,  2008,  1005,  1055,  2054,  1045,  2921,  4851,  2870,  2076,
         1996,  2116,  9590,  1010,  7491,  3503,  1010, 25082,  1998,  2236,
        26865,  2008,  2566,  4168,  3686,  1996,  6391,  2781,  1012,  1996,
        18539,  2036,  3233,  2039,  2043,  2017,  2228,  1997,  1996,  2028,
         1011,  8789,  3494,  1010,  2040,  2031,  2061,  2210,  5995,  2008,
         2009,  2003,  8990,  5263,  2000,  2729,  2054,  6433,  2000,  2068,
         1012,  2027,  2024,  2074,  6649,  2517, 22330, 27921,  2015,  2005,
         1996,  2472,  2000,  6865,  2010, 27135,  9029,  2006,  1010,  1037,
         8476,  2008,  2038,  2042,  2589,  2172,  2488,  1999,  2060, 16547,
         2119,  2006,  2694,  1998,  1996,  5988,  1012,  1026,  7987,  1013,
         1028,  1026,  7987,  1013,  1028,  1045,  2442, 18766,  1010,  1045,
         1005,  1049,  2025,  2428,  2028,  2005, 27963,  2919,  4616,  2076,
         1037,  2143,  1010,  2021,  2009,  2442, 

*Step 4: Fine-Tune the Model*

**Seting Up Training Parameters and training the model**

In [ ]:
# Define evaluation metrics
def compute_metrics(p):
    preds = p.predictions.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='weighted')
    accuracy = accuracy_score(p.label_ids, preds)

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-42-5c115ba4ac7d>:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.246200,0.215919,0.909000,0.910094,0.909000,0.908940
2,0.121800,0.227958,0.921200,0.921227,0.921200,0.921199


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=2500, training_loss=0.19787487335205078, metrics={'train_runtime': 1154.5878, 'train_samples_per_second': 69.289, 'train_steps_per_second': 2.165, 'total_flos': 5298695946240000.0, 'train_loss': 0.19787487335205078, 'epoch': 2.0})

In [ ]:
model.save_pretrained("fine-tuned-imdb_V2")
tokenizer.save_pretrained("fine-tuned-imdb_V2")

('fine-tuned-imdb_V2/tokenizer_config.json',
 'fine-tuned-imdb_V2/special_tokens_map.json',
 'fine-tuned-imdb_V2/vocab.txt',
 'fine-tuned-imdb_V2/added_tokens.json')

Log in to Hugging Face

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub("SeyedehMonaEbrahimi/fine-tuned-imdb_V2")
tokenizer.push_to_hub("SeyedehMonaEbrahimi/fine-tuned-imdb_V2")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/SeyedehMonaEbrahimi/fine-tuned-imdb_V2/commit/271721ec7e287d91c9597f77c636ac25767da72e', commit_message='Upload tokenizer', commit_description='', oid='271721ec7e287d91c9597f77c636ac25767da72e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SeyedehMonaEbrahimi/fine-tuned-imdb_V2', endpoint='https://huggingface.co', repo_type='model', repo_id='SeyedehMonaEbrahimi/fine-tuned-imdb_V2'), pr_revision=None, pr_num=None)



# **The Link to my HuggingFace_hub :https://huggingface.co/SeyedehMonaEbrahimi/fine-tuned-imdb**

# **Part 2: API Development and Testing**

Link to my Github including a complete Sentiment Analysis application with a FastAPI backend, React frontend, and a Jupyter Notebook for model finetuning.

# ** https://github.com/seyedeh-mona-ebrahimi/Fine-tuning_LLM_API/tree/main **



Link to my youtube video demo:

https://www.youtube.com/watch?v=vCUVF6ZgZXU